## Sudoku Solver
My attempt at creating algorithms to solve sudokus and to get better at understanding data structures and algorithms

Sudo-Code outline
OOP Method
- Define Sudoku class
- used np arrays or maybe tensors to store the puzzle?
- Define logic rules of the game
- Create recursive solving method

Function method
- idk by now

In [2]:
"""
GNN Sudoku Solver (pure PyTorch, no external GNN libs)

Now adapted for Jupyter Notebook usage:
- Removed argparse and __main__ block.
- Training parameters can be set as Python variables.
- Functions can be called interactively from cells.
"""
from __future__ import annotations
import random
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader


In [3]:

# ----------------------------
# Sudoku utilities
# ----------------------------

def sudoku_pattern(r: int, c: int) -> int:
    return (r * 3 + r // 3 + c) % 9

def shuffled(seq):
    seq = list(seq)
    random.shuffle(seq)
    return seq

def random_solved_board(seed: int | None = None) -> np.ndarray:
    if seed is not None:
        random.seed(seed)
    base = np.array([[sudoku_pattern(r, c) for c in range(9)] for r in range(9)], dtype=np.int64)
    digits = shuffled(range(9))
    board = digits[base]
    rows = sum((shuffled(range(g*3, g*3+3)) for g in shuffled(range(3))), [])
    cols = sum((shuffled(range(g*3, g*3+3)) for g in shuffled(range(3))), [])
    board = board[rows][:, cols]
    return board + 1

@dataclass
class Puzzle:
    puzzle: np.ndarray
    solution: np.ndarray

def make_puzzle(solution: np.ndarray, clues: int = 30, seed: int | None = None) -> Puzzle:
    if seed is not None:
        random.seed(seed)
    puzzle = solution.copy()
    n_remove = max(0, 81 - clues)
    positions = list(range(81))
    random.shuffle(positions)
    for idx in positions[:n_remove]:
        r, c = divmod(idx, 9)
        puzzle[r, c] = 0
    return Puzzle(puzzle=puzzle, solution=solution)

In [ ]:

# ----------------------------
# Graph construction
# ----------------------------

def build_adj_matrix() -> torch.Tensor:
    N = 81
    A = torch.zeros(N, N, dtype=torch.float32)
    def rc(i): return i // 9, i % 9
    def box_id(r, c): return (r // 3) * 3 + (c // 3)
    for i in range(N):
        ri, ci = rc(i)
        for j in range(N):
            if i == j: continue
            rj, cj = rc(j)
            if ri == rj or ci == cj or box_id(ri, ci) == box_id(rj, cj):
                A[i, j] = 1.0
    deg = A.sum(dim=1, keepdim=True).clamp(min=1.0)
    return A / deg

ROW_OH = torch.eye(9)
COL_OH = torch.eye(9)
BOX_OH = torch.eye(9)

def board_to_features(board: np.ndarray, given_mask: np.ndarray) -> torch.Tensor:
    feats = []
    for r in range(9):
        for c in range(9):
            b = (r // 3) * 3 + (c // 3)
            digit = board[r, c]
            digit_oh = torch.zeros(9)
            if digit > 0:
                digit_oh[digit - 1] = 1.0
            f = torch.cat([ROW_OH[r], COL_OH[c], BOX_OH[b], digit_oh, torch.tensor([1.0 if given_mask[r, c] else 0.0])])
            feats.append(f)
    return torch.stack(feats, dim=0)

def targets_from_solution(solution: np.ndarray) -> torch.Tensor:
    return torch.tensor([solution[r, c] - 1 for r in range(9) for c in range(9)], dtype=torch.long)

def mask_from_puzzle(puzzle: np.ndarray) -> torch.Tensor:
    return torch.tensor([puzzle[r, c] == 0 for r in range(9) for c in range(9)], dtype=torch.bool)

# ----------------------------
# Dataset
# ----------------------------

class SudokuStream(IterableDataset):
    def __init__(self, clues: int = 30, seed: int | None = None):
        self.clues = clues
        self.rng = random.Random(seed)
    def __iter__(self):
        while True:
            sol = random_solved_board(seed=self.rng.randrange(10**9))
            pz = make_puzzle(sol, clues=self.clues, seed=self.rng.randrange(10**9))
            given_mask = (pz.puzzle > 0)
            x = board_to_features(pz.puzzle, given_mask)
            y = targets_from_solution(pz.solution)
            loss_mask = mask_from_puzzle(pz.puzzle)
            yield x, y, loss_mask

# ----------------------------
# Model
# ----------------------------

class MPNNLayer(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.lin_self = nn.Linear(dim, dim)
        self.lin_msg = nn.Linear(dim, dim)
        self.lin_update = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
    def forward(self, h: torch.Tensor, A_norm: torch.Tensor) -> torch.Tensor:
        m_self = self.lin_self(h)
        m_nei = A_norm @ h
        m_nei = self.lin_msg(m_nei)
        u = F.relu(self.norm(self.lin_update(m_self + m_nei)))
        return h + u

class SudokuGNN(nn.Module):
    def __init__(self, in_dim: int = 37, hidden: int = 256, layers: int = 6):
        super().__init__()
        self.enc = nn.Linear(in_dim, hidden)
        self.layers = nn.ModuleList([MPNNLayer(hidden) for _ in range(layers)])
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 9),
        )
    def forward(self, x: torch.Tensor, A_norm: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.enc(x))
        for layer in self.layers:
            h = layer(h, A_norm)
        return self.head(h)

# ----------------------------
# Training / Evaluation
# ----------------------------

@dataclass
class Batch:
    x: torch.Tensor
    y: torch.Tensor
    m: torch.Tensor

def collate(batch_list: List[Tuple[torch.Tensor, torch.Tensor, torch.Tensor]]) -> Batch:
    xs, ys, ms = zip(*batch_list)
    return Batch(torch.stack(xs, dim=0), torch.stack(ys, dim=0), torch.stack(ms, dim=0))

def train_one_epoch(model, loader, A_norm, opt, device, log_every=100):
    model.train()
    ce = nn.CrossEntropyLoss(reduction='none')
    total_loss, total_tokens = 0.0, 0
    for step, batch in enumerate(loader, start=1):
        x, y, m = batch.x.to(device), batch.y.to(device), batch.m.to(device)
        B = x.size(0)
        opt.zero_grad()
        logits = model(x.view(B*81, -1), A_norm.repeat(B, 1, 1)).view(B,81,9)
        loss_all = ce(logits.view(B*81, 9), y.view(B*81))
        loss = (loss_all.view(B,81)[m]).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        n = m.sum().item()
        total_loss += loss.item() * n
        total_tokens += n
        if step % log_every == 0:
            print(f"  step {step}: loss={loss.item():.4f} on {n} masked cells")
    return total_loss / max(1, total_tokens)

def evaluate(model, n_batches, clues, device):
    model.eval()
    ds = SudokuStream(clues=clues, seed=123)
    loader = DataLoader(ds, batch_size=32, collate_fn=collate)
    A = build_adj_matrix().to(device)
    correct, total, solved = 0, 0, 0
    with torch.no_grad():
        for _ in range(n_batches):
            x, y, m = next(iter(loader))
            x, y = x.to(device), y.to(device)
            B = x.size(0)
            logits = model(x.view(B*81, -1), A.repeat(B,1,1)).view(B,81,9)
            pred = logits.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += y.numel()
            solved += ((pred == y).all(dim=1)).sum().item()
    return correct / total, solved / (n_batches * 32)

# ----------------------------
# Inference helper
# ----------------------------

def solve_with_model(model, puzzle: np.ndarray, device: str = 'cpu') -> np.ndarray:
    model.eval()
    A = build_adj_matrix().to(device)
    given_mask = (puzzle > 0)
    x = board_to_features(puzzle, given_mask).to(device)
    with torch.no_grad():
        pred = model(x, A).argmax(dim=-1).cpu().numpy() + 1
    return pred.reshape(9,9)

# ----------------------------
# Example usage in notebook
# ----------------------------

# Set parameters directly in notebook
EPOCHS = 2
BATCH_SIZE = 64
HIDDEN = 256
LAYERS = 6
CLUES = 30
LR = 1e-3
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Prepare data, model, optimizer, scheduler
A_norm = build_adj_matrix().to(DEVICE)
ds = SudokuStream(clues=CLUES, seed=42)
loader = DataLoader(ds, batch_size=BATCH_SIZE, collate_fn=collate)

model = SudokuGNN(in_dim=37, hidden=HIDDEN, layers=LAYERS).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}")
    avg_loss = train_one_epoch(model, loader, A_norm, opt, DEVICE, log_every=50)
    acc, solved = evaluate(model, n_batches=2, clues=CLUES, device=DEVICE)
    scheduler.step()
    print(f"Epoch {epoch} done. avg_loss={avg_loss:.4f} eval_acc={acc:.4f} solved_rate={solved:.3f}")

# Demo
sol = random_solved_board()
pz = make_puzzle(sol, clues=CLUES)
pred = solve_with_model(model, pz.puzzle, device=DEVICE)

def pretty(b):
    return "\n".join(" ".join(str(x) for x in row) for row in b)

print("\nPuzzle:\n" + pretty(pz.puzzle))
print("\nPrediction:\n" + pretty(pred))
print("\nSolution:\n" + pretty(pz.solution))
